# Calculate the geometric delay between TART stations

- Load the acquired satellite positions
- Calculate the inter-TART delays from satellite to TART sites
- Save delays to disk

In [1]:
import numpy as np
import sys
import os
import json
# Get parent directory
parent_dir = os.path.abspath("..")
# Add it to sys.path
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

from pipeline.GPS import GPS_handler
from pipeline.JSON_handler import load_from_json, to_json_safe, save_to_json
from pipeline.acquisition_handler import acquire_GPS 
from pipeline.geometric_delay import geometry

## Load the required data from previous stages

- Viable GPS positions
- Viable GPS names
- GPS doppler acquisition results
- TART positions

In [2]:
TARTS = {
    'namibia': 'na-unam',
    'rhodes': 'za-rhodes'
}

t_obs = 1770993565.3405828

obs_filepath = f"/home/jdawson/repos/TART/notebooks/dataproducts/{t_obs}"
meta_filepath = f"/home/jdawson/repos/TART/datasets/GPS"

GPS_positions_filename = "GPS_positions.json"
GPS_positions = load_from_json(obs_filepath, GPS_positions_filename)

viable_satellites_filename = "GPS_viable_satellites.json"
viable_satellites = load_from_json(obs_filepath, viable_satellites_filename)

acquisition_results_filename = 'GPS_acquisition_results.json'
acquisition_results = load_from_json(obs_filepath, acquisition_results_filename)

TART_positions_filename = "TART_coords.json"
TART_positions = load_from_json(meta_filepath, TART_positions_filename)

## Get the union of viable satellites between the two selected TART sites above

In [3]:
handler = GPS_handler()
viable_satellites_union = handler.get_union_satellites(viable_satellites, TARTS)
print(viable_satellites_union)

['G10', 'G23', 'G25', 'G28', 'G31', 'G32']


## Filter the viable satellites based on their acquisition SNRs

In [4]:
acquisition_handler = acquire_GPS()
filtered_satellites = acquisition_handler.filter_acquired_satellites(acquisition_results, viable_satellites_union, TARTS)
print(filtered_satellites)

['G23' 'G28' 'G31']


Take a look a the doppler acquisition SNRs for each antenna for each satellite

In [5]:
acquisition_tables = acquisition_handler.snr_tables(acquisition_results, filtered_satellites, list(TARTS.keys()))

,Ant 0,Ant 1,Ant 2,Ant 3,Ant 4,Ant 5,Ant 6,Ant 7,Ant 8,Ant 9,Ant 10,Ant 11,Ant 12,Ant 13,Ant 14,Ant 15,Ant 16,Ant 17,Ant 18,Ant 19,Ant 20,Ant 21,Ant 22,Ant 23
namibia,15.71,22.60,39.27,22.91,25.39,18.51,30.29,22.65,11.83,29.20,14.98,23.10,16.55,27.44,22.60,28.18,33.68,38.84,36.77,20.59,21.95,15.93,13.13,14.80
rhodes,35.30,23.52,30.72,6.59,4.01,2.19,41.84,7.39,59.12,28.55,2.85,34.73,39.86,31.93,1.73,25.60,31.41,37.10,2.11,4.14,2.59,2.77,31.81,40.92


,Ant 0,Ant 1,Ant 2,Ant 3,Ant 4,Ant 5,Ant 6,Ant 7,Ant 8,Ant 9,Ant 10,Ant 11,Ant 12,Ant 13,Ant 14,Ant 15,Ant 16,Ant 17,Ant 18,Ant 19,Ant 20,Ant 21,Ant 22,Ant 23
namibia,8.45,19.49,10.94,11.88,17.13,13.02,10.26,14.81,25.24,23.93,8.19,24.11,14.37,12.80,9.79,20.34,18.81,10.14,23.96,13.87,5.58,15.95,6.80,12.47
rhodes,15.42,23.36,29.04,2.03,2.00,2.07,20.61,3.43,20.40,26.04,2.50,17.72,15.61,20.63,1.73,19.79,16.51,7.59,1.99,2.03,2.00,2.06,20.32,23.07


,Ant 0,Ant 1,Ant 2,Ant 3,Ant 4,Ant 5,Ant 6,Ant 7,Ant 8,Ant 9,Ant 10,Ant 11,Ant 12,Ant 13,Ant 14,Ant 15,Ant 16,Ant 17,Ant 18,Ant 19,Ant 20,Ant 21,Ant 22,Ant 23
namibia,14.03,45.08,29.56,21.51,20.26,21.42,19.73,21.72,10.93,27.38,7.09,26.54,22.82,23.82,24.61,26.63,20.62,16.23,27.72,18.65,12.06,18.76,9.27,12.45
rhodes,29.28,13.68,15.20,5.96,4.09,2.33,22.36,9.76,33.70,27.74,7.15,15.40,17.26,16.93,1.73,20.85,10.68,26.53,2.13,6.09,5.92,5.23,14.60,12.20


## Declare reference antennas from each site

In [6]:
ref_antennas = {
    "G23": {
        "rhodes": 8,
        "namibia": 2,
    },
    "G28": {
        "rhodes": 2,
        "namibia": 8,
    },
    "G31": {
        "rhodes": 8,
        "namibia": 1,
    }
}

## Calculate the geometric delay between antennas

In [7]:
# Get the TART antenna positions into a dict

TART_antenna_positions = {}
for TART in TARTS.keys():
    TART_antenna_positions[TART] = np.array(TART_positions[TART]["antennas_ecef"])

In [8]:
# Get the positions of the filtered satellites

filtered_GPS_positions = {
    k: v for k, v in GPS_positions.items() if k in filtered_satellites
}

In [14]:
geometry_handler = geometry()

delays = geometry_handler.compute_inter_tart_delays(
    visible_satellites=filtered_GPS_positions,
    tart_positions=TART_antenna_positions,
    tart_ref="rhodes",
    ref_antennas=ref_antennas,
)

print("These are the time delays that need to be applied to TARTs in order to shift them to the frame of the 'tart_ref' chosen in compute_inter_tart_delays()")
print(delays)

These are the time delays that need to be applied to TARTs in order to shift them to the frame of the 'tart_ref' chosen in compute_inter_tart_delays()
{'G23': {'namibia': 0.002051478955840275}, 'G28': {'namibia': 0.001105966175862335}, 'G31': {'namibia': 0.003299716710206912}}


In [12]:
serialisable = to_json_safe(delays)
filename = f"inter_TART_geometric_delays.json"
save_to_json(serialisable, obs_filepath, filename)

'JSON saved to disk.'